<a href="https://colab.research.google.com/github/anushah-200/factcheck_AI_model_agnostic/blob/main/notebooks/12_LOMO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
BASE = "/content/drive/MyDrive/factcheckAI/outputs/"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
df = pd.read_csv(BASE + "factcheck_clean_dataset.csv")
print(df.shape)
df["Model"].value_counts()

(806, 19)


,count
Model,
OpenAI,279
DeepSeek,270
Groq,257


In [18]:
CAT_FEATURES = ["Category", "Type"]
NUM_FEATURES = ["ResponseLength", "QuestionLength", "ResponseCharacters", "AverageWordLength"]
TARGET = "Hallucination"

LOMO_EXPERIMENTS = [
    (["OpenAI", "Groq"], "DeepSeek"),
    (["OpenAI", "DeepSeek"], "Groq"),
    (["Groq", "DeepSeek"], "OpenAI"),
]

CLASSIFIERS = {
    "LogisticRegression": lambda: LogisticRegression(max_iter=1000, random_state=42),
    "DecisionTree": lambda: DecisionTreeClassifier(random_state=42),
    "RandomForest": lambda: RandomForestClassifier(n_estimators=100, random_state=42),
}

In [19]:
def build_features(train_df, test_df):
    cat_features = ["Category", "Type"]

    num_features = [
        "ResponseLength",
        "QuestionLength",
        "ResponseCharacters",
        "AverageWordLength"
    ]

    preprocessor = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
            ("num", "passthrough", num_features)
        ]
    )

    X_train = train_df[cat_features + num_features]
    X_test = test_df[cat_features + num_features]

    X_train_transformed = preprocessor.fit_transform(X_train)
    X_test_transformed = preprocessor.transform(X_test)

    return X_train_transformed, X_test_transformed

In [20]:
def evaluate(clf, X_test, y_test):
    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)[:, 1]
    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, y_proba),
        "confusion_matrix": confusion_matrix(y_test, y_pred).tolist(),
    }

In [21]:
all_results = []

for train_models, test_model in LOMO_EXPERIMENTS:

    train_df = df[df["Model"].isin(train_models)].copy()
    test_df = df[df["Model"] == test_model].copy()

    X_train, X_test = build_features(train_df, test_df)

    y_train = train_df[TARGET]
    y_test = test_df[TARGET]

    print(
        f"\n{'='*60}\n"
        f"Train {train_models} (n={len(train_df)}) "
        f"-> Test {test_model} (n={len(test_df)})\n"
        f"{'='*60}"
    )

    for clf_name, clf_factory in CLASSIFIERS.items():

        clf = clf_factory()

        clf.fit(X_train, y_train)

        metrics = evaluate(
            clf,
            X_test,
            y_test
        )

        print(
            f"\n{clf_name}: "
            f"Acc={metrics['Accuracy']:.4f}  "
            f"F1={metrics['F1']:.4f}  "
            f"ROC-AUC={metrics['ROC_AUC']:.4f}"
        )

        all_results.append({
            "Train": "+".join(train_models),
            "Test": test_model,
            "Classifier": clf_name,
            **{
                k: v
                for k, v in metrics.items()
                if k != "confusion_matrix"
            },
        })


Train ['OpenAI', 'Groq'] (n=536) -> Test DeepSeek (n=270)


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



LogisticRegression: Acc=0.8074  F1=0.7984  ROC-AUC=0.8745

DecisionTree: Acc=0.8185  F1=0.8151  ROC-AUC=0.8234

RandomForest: Acc=0.8185  F1=0.8137  ROC-AUC=0.8589

Train ['OpenAI', 'DeepSeek'] (n=549) -> Test Groq (n=257)


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



LogisticRegression: Acc=0.8560  F1=0.8795  ROC-AUC=0.9280

DecisionTree: Acc=0.7977  F1=0.8312  ROC-AUC=0.7940

RandomForest: Acc=0.8949  F1=0.9132  ROC-AUC=0.9495

Train ['Groq', 'DeepSeek'] (n=527) -> Test OpenAI (n=279)

LogisticRegression: Acc=0.8566  F1=0.8750  ROC-AUC=0.9344

DecisionTree: Acc=0.8315  F1=0.8479  ROC-AUC=0.8328

RandomForest: Acc=0.8674  F1=0.8825  ROC-AUC=0.9375


In [22]:
for train_models, test_model in LOMO_EXPERIMENTS:
    train = df[df["Model"].isin(train_models)]
    test = df[df["Model"] == test_model]
    print(f"Train {train_models}: {train['Hallucination'].value_counts(normalize=True).round(3).to_dict()}")
    print(f"Test  {test_model}: {test['Hallucination'].value_counts(normalize=True).round(3).to_dict()}\n")

Train ['OpenAI', 'Groq']: {1: 0.591, 0: 0.409}
Test  DeepSeek: {0: 0.544, 1: 0.456}

Train ['OpenAI', 'DeepSeek']: {1: 0.514, 0: 0.486}
Test  Groq: {1: 0.615, 0: 0.385}

Train ['Groq', 'DeepSeek']: {1: 0.533, 0: 0.467}
Test  OpenAI: {1: 0.57, 0: 0.43}



In [23]:
results_df = pd.DataFrame(all_results)
results_df.to_csv(BASE + "lomo_results.csv", index=False)

rf_summary = results_df[results_df["Classifier"] == "RandomForest"][
    ["Train", "Test", "Accuracy", "Precision", "Recall", "F1", "ROC_AUC"]
]
print(rf_summary.to_string(index=False))

          Train     Test  Accuracy  Precision   Recall       F1  ROC_AUC
    OpenAI+Groq DeepSeek  0.818519   0.764286 0.869919 0.813688 0.858857
OpenAI+DeepSeek     Groq  0.894942   0.928105 0.898734 0.913183 0.949463
  Groq+DeepSeek   OpenAI  0.867384   0.891026 0.874214 0.882540 0.937500
